<a href="https://colab.research.google.com/github/nanaaries313/Portfolio/blob/main/Boosted_Trees_using_xgboost_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # **Week 12: Boosted trees using xgboost()**

In [2]:
pkgs <- c("xgboost")
to_install <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(to_install) > 0) install.packages(to_install)

In [3]:
library(xgboost)

We'll use the glass data again


In [5]:
pkgs <- c("mlbench")
to_install <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(to_install) > 0) install.packages(to_install)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [6]:
data(Glass, package="mlbench")
str(Glass)

'data.frame':	214 obs. of  10 variables:
 $ RI  : num  1.52 1.52 1.52 1.52 1.52 ...
 $ Na  : num  13.6 13.9 13.5 13.2 13.3 ...
 $ Mg  : num  4.49 3.6 3.55 3.69 3.62 3.61 3.6 3.61 3.58 3.6 ...
 $ Al  : num  1.1 1.36 1.54 1.29 1.24 1.62 1.14 1.05 1.37 1.36 ...
 $ Si  : num  71.8 72.7 73 72.6 73.1 ...
 $ K   : num  0.06 0.48 0.39 0.57 0.55 0.64 0.58 0.57 0.56 0.57 ...
 $ Ca  : num  8.75 7.83 7.78 8.22 8.07 8.07 8.17 8.24 8.3 8.4 ...
 $ Ba  : num  0 0 0 0 0 0 0 0 0 0 ...
 $ Fe  : num  0 0 0 0 0 0.26 0 0 0 0.11 ...
 $ Type: Factor w/ 6 levels "1","2","3","5",..: 1 1 1 1 1 1 1 1 1 1 ...


## **Preparing the dependent variables for xgboost()**

No longer needed!

xgboost() requries the dependent variable to be an integer starting from zero (or continuous). We therefore need to convert factors to the integers starting at zero.

For any data where the dependent variable is a class (factor), do this in two steps:

STEP 1. First save the old class labels so you can get back to them later after you build a model and make predictions.(This is not such a big deal here because the labels are just "1,2,..." but in general you would want to recover them.)

In [7]:
glass.type = Glass$Type

STEP 2: Then create new class labels

In [29]:
x.type = as.integer(Glass$Type)-1

## **Format the data in a xbg.DMatrix**

xgboost() has their own data structure: xbg.DMatrix (not using data frame)

Split into test-train (resampling may be more appropriate for such small data, but that's not our focus today)

First create a subset of indices

In [9]:
n=nrow(Glass)
set.seed(1234)
test = sample(1:n, as.integer(n/3))

xgboost() is also picky about the format for the input data. You want to create special type of matrix format.

 We need two conversions from our standard data frame:

  a) data frame -> matrix

  b) matrix -> xgb.DMatrix

For a) we use the base as.matrix() function. For b) we use the xgb.DMatrix() function, which inputs the ordinary matrix with the independent variables only and the class labels that we already constructed.

Create the training data:

In [10]:
train.input = xgb.DMatrix(as.matrix(Glass[-test,-10]),
                          label=x.type[-test])

*Remove the 10th column (Type- factor values), replaced with new one (Type - integers)*

Create the test data

In [11]:
test.input = xgb.DMatrix(as.matrix(Glass[test,-10]),
                        label=x.type[test])

 Now we have the correctly formated data to input into the xgboost() function!

In [12]:
str(train.input)

Class 'xgb.DMatrix' <externalptr> 
 - attr(*, "fields")=<environment: 0x55c191cddc98> 


## **Train a boosted tree**

 There is a huge number of parameters required for xgboost()

 Old implementation: list; new implementation: parameters now are functions

 The following shows how to set some:

In [13]:
params = list(
  booster="gbtree",    # Base learner (options include two trees and one linear)

  #Main regularization
  eta=0.3,                   # Step-size shrinkage

  #Controling tree size (not advise to change this)
  max_depth=6,                 # Max tree depth
  gamma=0,                     # Minimum loss for split

  #  Adding randomness to learning trees (1=no sampling, no recommend to change)
  subsample=1,                 # Sample of data used
                               # for each tree

  colsample_bytree=1,          # Sample of variables
                               # used by each tree
  colsample_bylevel=1,         # Another way to do the same!
  colsample_bynode=1,          # Another way to do the same!

  #Other regularization (by default, use L2, not L1)
  lambda = 1,                  # L2 regularization (shrinkage)
  alpha = 0,                   # L1 regularization (shrinkage)

  #Dependent variable
  objective="multi:softprob",  # Type of dependent variable
  eval_metric="merror",        # "error", "rmse", "mlogloss"
  num_class = length(levels(glass.type))
  )


---

  *   gbtree: gradient boosted tree
  *   eta: Control how much each new tree's predictions contribute to the final model, preventing overfitting. Add new error to the previous trees. (0.01-0.3)
       *   High: larger impact on the model. easy to overfit
       *   Low: smaller impact

  *   subsample = 0.8 means 80% of the data

  *   Objective = "multi:softprob": This defines the learning task and the corresponding objective function. Multi:softprob is used for multi-class classifications (different types of glass)
  *   merror: multi-class error
  *   num_class = length (levels (glass.type)): This sets the number of target classes, which determined by the number of unique levels
  ---

Gamma is similar to cp. Set to higher value to make sure it don't build to complex tree

The above are the default parameters. So by default xgboost does double shrinkage (step size and L2 = step direction),but does not use sampling.

For a full list of parameters see: https://xgboost.readthedocs.io/en/latest/parameter.html


Now training the model is simple. Because we put a lot of effort into preparing the input data and setting parameters!

The only thing left is to set the number of trees created,that is the number of boosting iterations.

Old: xgboost() -> new: xgb.train(data, target variables, parameters)

In [14]:
boost.model = xgb.train (data=train.input,  # Training data
                      nrounds = 1000,       # Boosting iterations
                      params=params)        # Parameters

Notice that this was very fast.

With a model, we can now make predictions (almost) as usual:

In [15]:
glass.predictions = predict(boost.model,test.input,
                            reshape=T)   # What's this reshape?
str(glass.predictions)

Warning message in throw_err_or_depr_msg("Parameter(s) have been removed from this function: ", :
“Parameter(s) have been removed from this function: reshape. This warning will become an error in a future version.”


 num [1:71, 1:6] 0.99933 0.0455 0.84697 0.10765 0.00261 ...


xgboost ouputs are a flat vector of probabilities. Reshape organizes these probabilities into matrix (no need anymore)

Data shape is either long or wide. We have only worked with wide data in this class, but sometimes working with long data is faster.

The predict() function for xgboost has a lot of options.Find them by typing ?predict.xgb.Booster

The output for the predictions is a matrix, just like the input. The xgboost() function just doesn't do data frames!

So we'll convert it to a data frame ourselves:

In [16]:
glass.predictions = as.data.frame(glass.predictions)
str(glass.predictions)

'data.frame':	71 obs. of  6 variables:
 $ V1: num  0.99933 0.0455 0.84697 0.10765 0.00261 ...
 $ V2: num  0.000351 0.951299 0.127361 0.674408 0.98116 ...
 $ V3: num  0.000156 0.000737 0.014447 0.110005 0.001771 ...
 $ V4: num  6.32e-05 1.22e-03 3.92e-03 4.02e-02 6.59e-03 ...
 $ V5: num  5.46e-05 4.73e-04 3.37e-03 2.90e-02 2.87e-03 ...
 $ V6: num  4.13e-05 7.74e-04 3.92e-03 3.87e-02 5.00e-03 ...


Go back to the original column names instead of default column names

In [17]:
colnames(glass.predictions) = levels(glass.type)
str(glass.predictions)

'data.frame':	71 obs. of  6 variables:
 $ 1: num  0.99933 0.0455 0.84697 0.10765 0.00261 ...
 $ 2: num  0.000351 0.951299 0.127361 0.674408 0.98116 ...
 $ 3: num  0.000156 0.000737 0.014447 0.110005 0.001771 ...
 $ 5: num  6.32e-05 1.22e-03 3.92e-03 4.02e-02 6.59e-03 ...
 $ 6: num  5.46e-05 4.73e-04 3.37e-03 2.90e-02 2.87e-03 ...
 $ 7: num  4.13e-05 7.74e-04 3.92e-03 3.87e-02 5.00e-03 ...


We can pick out the most likely value using the which.max() function (find the index, row number, of highest probability for each column)

In [18]:
which.max(glass.predictions[,1]) # etc

[1] 60

We want to repeat this for the other 70 data points, which could be done in a for-loop or an apply function

Create a new column with the most likely class value

In [23]:
glass.predictions$prediction = apply(glass.predictions,1,
                                     function(x) colnames(glass.predictions)[which.max(x)])

*Appy function(x)-Find the row with the largest probability and retrieve the column name from it, on glass prediction, with a margin = 1 (row by row)*

Create a new column with the actual class label

In [20]:
glass.predictions$label = levels(glass.type)[x.type[test]+1]
str(glass.predictions)

'data.frame':	71 obs. of  8 variables:
 $ 1         : num  0.99933 0.0455 0.84697 0.10765 0.00261 ...
 $ 2         : num  0.000351 0.951299 0.127361 0.674408 0.98116 ...
 $ 3         : num  0.000156 0.000737 0.014447 0.110005 0.001771 ...
 $ 5         : num  6.32e-05 1.22e-03 3.92e-03 4.02e-02 6.59e-03 ...
 $ 6         : num  5.46e-05 4.73e-04 3.37e-03 2.90e-02 2.87e-03 ...
 $ 7         : num  4.13e-05 7.74e-04 3.92e-03 3.87e-02 5.00e-03 ...
 $ prediction: chr  "1" "2" "1" "2" ...
 $ label     : chr  "1" "2" "3" "2" ...


Now a confusion matrix is easy

In [21]:
CM = table(glass.predictions$prediction,
           glass.predictions$label)
CM

   
     1  2  3  5  6  7
  1 18  8  2  0  0  0
  2  2 21  0  1  0  0
  3  1  1  3  0  0  0
  5  0  1  0  3  0  0
  6  0  0  0  0  3  0
  7  0  0  0  1  1  5

And we can calculate the error

In [22]:
error = 1 - sum(glass.predictions$prediction==glass.predictions$label)/nrow(glass.predictions)
error

[1] 0.2535211

This is pretty good but you are likely able to tune it to get something better.

What's your concern?